In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf



# Deskriptiv analyse

In [3]:
df = pd.read_stata('Integration.dta')
df.columns
# Får alle dummies og laver den til binær. 
year_dummies = pd.get_dummies(df['ankomstaar'], prefix='y').astype(int)


df = pd.concat([df, year_dummies], axis=1)

var = ['aarslon', 'alder'] + year_dummies.columns.tolist()

desc = df.groupby('mand')[var].agg(['mean', 'std'])

table = pd.DataFrame(index=var, columns=['Kvinder', 'Mænd'])

for v in var:
    table.loc[v, 'Kvinder'] = f"{desc.loc[0, (v, 'mean')]:.3f}"
    table.loc[v, 'Mænd'] = f"{desc.loc[1, (v, 'mean')]:.3f}"
    
    table.loc[v + '_std', 'Kvinder'] = f"({desc.loc[0, (v, 'std')]:.3f})"
    table.loc[v + '_std', 'Mænd'] = f"({desc.loc[1, (v, 'std')]:.3f})"

print(table.to_latex(escape=False))

df.describe()


\begin{tabular}{lll}
\toprule
 & Kvinder & Mænd \\
\midrule
aarslon & 35584.742 & 76115.273 \\
alder & 29.128 & 29.170 \\
y_2014.0 & 0.341 & 0.333 \\
y_2015.0 & 0.335 & 0.317 \\
y_2016.0 & 0.323 & 0.351 \\
aarslon_std & (21035.548) & (46778.676) \\
alder_std & (6.978) & (7.083) \\
y_2014.0_std & (0.474) & (0.471) \\
y_2015.0_std & (0.472) & (0.465) \\
y_2016.0_std & (0.468) & (0.477) \\
\bottomrule
\end{tabular}



,id,mand,ankomstaar,alder,aarslon,aar,y_2014.0,y_2015.0,y_2016.0
count,4800.000000,4800.000000,4800.000000,4800.000000,4800.000000,4800.000000,4800.000000,4800.000000,4800.000000
mean,1200.500000,0.793750,2015.010376,29.161667,67755.851562,2017.500000,0.334583,0.320417,0.345000
std,692.892456,0.404654,0.824389,7.060873,45793.058594,0.500052,0.471894,0.466685,0.475418
min,1.000000,0.000000,2014.000000,17.000000,4526.069824,2017.000000,0.000000,0.000000,0.000000
25%,600.750000,1.000000,2014.000000,24.000000,38856.023438,2017.000000,0.000000,0.000000,0.000000
50%,1200.500000,1.000000,2015.000000,28.000000,57261.664062,2017.500000,0.000000,0.000000,0.000000
75%,1800.250000,1.000000,2016.000000,33.000000,83213.689453,2018.000000,1.000000,1.000000,1.000000
max,2400.000000,1.000000,2016.000000,66.000000,602016.812500,2018.000000,1.000000,1.000000,1.000000


# Regression OLS

#### 1.b

In [4]:
# Laver ny dataframe, hvor kun er fra 2017
df = df[df['aar'] == 2017.0]

D_year = year_dummies.columns.tolist()
#Fjerner referance kategorien 2014
D_year.remove("y_2014.0")

df['log_aarslon'] = np.log(df['aarslon'])

df['YSM'] = df['aar'] - df['ankomstaar']

result = smf.ols('log_aarslon ~ alder + YSM', data=df).fit()

print(result.summary())


                            OLS Regression Results                            
Dep. Variable:            log_aarslon   R-squared:                       0.032
Model:                            OLS   Adj. R-squared:                  0.031
Method:                 Least Squares   F-statistic:                     39.84
Date:                Tue, 06 Oct 2026   Prob (F-statistic):           9.53e-18
Time:                        10:49:42   Log-Likelihood:                -1639.8
No. Observations:                2400   AIC:                             3286.
Df Residuals:                    2397   BIC:                             3303.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     10.4231      0.047    219.799      0.0

##### 1.c

In [9]:
df['alder_ankomst'] = df['alder'] - df['YSM']


result2 = smf.ols('log_aarslon ~ alder_ankomst + YSM', data=df).fit()

print(result2.summary())


                            OLS Regression Results                            
Dep. Variable:            log_aarslon   R-squared:                       0.032
Model:                            OLS   Adj. R-squared:                  0.031
Method:                 Least Squares   F-statistic:                     39.84
Date:                Tue, 06 Oct 2026   Prob (F-statistic):           9.53e-18
Time:                        10:50:01   Log-Likelihood:                -1639.8
No. Observations:                2400   AIC:                             3286.
Df Residuals:                    2397   BIC:                             3303.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        10.4231      0.047    219.799